# Julia notebook to compute the symbolic solution to the buoyancy equation with a specified relaxation buoyancy field.

twnh Dec '25, Jul '26

This notebook solves

\begin{align}
 \epsilon^2   \kappa \frac{d^2 b}{d z^2} + \gamma ( B(z) - b ) = 0
\end{align}
for $b(z)$, given diffusivity $\kappa = \kappa_0$, and boundary conditions
\begin{align}
\text{Surface~}z = 0: b &= 0 ,\\
\text{Bottom~}z = -H: \frac{\partial b}{\partial z} &= 0
\end{align}
where $B(z)$ is the specified relaxation buoyancy field and $\gamma$ is the relaxation rate.

This version builds on `ExampleSolution_v_0.5.ipynb` but removes the numerical calculations. This code includes several checks to verify explicit analytical forms (in the LaTeX notes, `PlanetaryGeostrophySolutions_v0.5.tex`) match the SymPy calculations.

In [ ]:
using SymPy
im = SymPy.im  # SymPy's imaginary unit

### Define symbols:

In [ ]:
# Geometry symbolic parameters:
z, ξ   = symbols("z ξ",   real=true, negative=true) # Vertical coordinate and source location (both in [-H,0])
x, y   = symbols("x y",   real=true)                # Horizontal coordinates
H      = SymFunction("H", real=true, positive=true) # Domain depth H(x,y)
geometry_params = (H(x,y), z, ξ)

# Frictional thermal wind equation symbolic parameters:
f, ϵ   = symbols("f ϵ",   real=true, positive=true) # Coriolis parameter and Ekman number
ν₀, ϕ  = symbols("ν₀ ϕ",  real=true, positive=true) # Viscosity parameters
τs     = SymFunction("τs",    complex=true)         # Complex surface wind stress
psg    = SymFunction("psg",   complex=true)         # Surface pressure gradient (∂/∂x + i ∂/∂y) pₛ(x,y)
b      = SymFunction("b",     real=true)            # Buoyancy field b(x,y,z)

# Define symbolic viscosity here:
ν = ν₀                                              # Constant viscosity profile
uv_params = (f, ϵ, ν) 

# Buoyancy equation symbolic parameters:
κ₀, ψ  = symbols("κ₀ ψ",  real=true, positive=true) # Diffusivity parameters
γ, α   = symbols("γ α",   real=true, positive=true) # Relaxation parameter
B      = SymFunction("B", real=true)                # Relaxation buoyancy field

# Define diffusivity profile here:
κ = κ₀                                              # Constant viscosity profile

b_params = (κ, ϵ, γ) 

# Convenience variables:
root_iϕ = ϕ*sympy.sqrt(im)
sample_params = Dict(ν₀=>1//2, κ₀=>1, ϵ=>2, ψ=>3, root_iϕ=>4im, ϕ => 4, H(x,y)=>1 - x^2 - y^2)
sample_point  = Dict(x=>-0.6, y=>0.2, z=>-0.32)

Dict{Sym{PyCall.PyObject}, Float64} with 3 entries:
  z => -0.32
  x => -0.6
  y => 0.2

### Compute pressure fields for the specified buoyancy field $b$:

In [4]:
# Define buoyancy gradient field:
bg     = SymFunction("bg",    complex=true)         # Buoyancy gradient

# Compute pressure field:
pbarog = SymPy.integrate(bg(x,y,ξ),(ξ,z,0))         # Baroclinic presure gradient 
pg     = psg(x,y) + pbarog                          # Total pressure gradient
pbotg  = pg.subs(z,-H(x,y))                         # Bottom pressure gradient

               0                   
               ⌠                   
psg(x, y) +    ⎮     bg(x, y, ξ) dξ
               ⌡                   
            -H(x, y)               

### Function to solve the frictional geostrophic equation using a Green's function:

In [5]:
function compute_Guv(uv_params, geometry_params)
    # Setup symbols and parameters:
    f, ϵ, ν = uv_params
    H, z, ξ = geometry_params
    uv      = SymFunction("uv")
    A       = symbols("A",real=true)                                                  # Unknown coefficient in the Green's function solution

    # 0. Define the ODE for d/dz(uv(z)) = uv(z):
    ode = Eq(-im * f * uv(z) + ϵ^2 * diff(diff(ν * uv(z),z),z), 0)
    
    # 1. Solve for $G_- (z)$ on $-H \le z \le ξ$:
    Gₘ = dsolve(ode, uv(z), ics = Dict(uv(z).subs(z,-H(x,y))=>0)).rhs
    @assert sympy.simplify(ode.lhs.subs(uv(z),Gₘ)) == 0                               # Check solution
    # Replace constant names because otherwise they can interfere with the constants from the next dsolve below.
    const_names = collect([string(s) for s in Gₘ.free_symbols if occursin(r"^C\d+", string(s))])
    Gₘ = Gₘ.subs(const_names[1],A)
    @assert sympy.simplify(Gₘ.subs(z,-H(x,y))) == 0                                   # Check bottom BC

    # 2. Solve for $G_+(z)$ on  $ξ \le z \le 0$:
    Gₚ = dsolve(ode, uv(z), ics = Dict(diff(uv(z),z).subs(z,0)=>0)).rhs
    @assert sympy.simplify(ode.lhs.subs(uv(z),Gₚ)) == 0                               # Check solution
    @assert sympy.simplify(diff(Gₚ,z).subs(z,0)) == 0                                 # Check surface BC

    # 3. Compute Wronskian $W(z)$:
    W = Gₘ * diff(Gₚ, z) - Gₚ * diff(Gₘ, z)

    # 4. Compute Green's function $G(z; \xi)$:
    Gm = Gₘ * Gₚ.subs(z,ξ) / (ϵ^2 * ν.subs(z,ξ) * W.subs(z,ξ))
    Gp = Gₘ.subs(z,ξ) * Gₚ / (ϵ^2 * ν.subs(z,ξ) * W.subs(z,ξ))

    # Check continuity and jump condition:
    @assert Gm.subs(z,ξ) - Gp.subs(z,ξ) == 0
    @assert sympy.simplify(diff(Gm, z).subs(z,ξ) - diff(Gp, z).subs(z,ξ)) + 1/(ϵ^2 * ν.subs(z,ξ)) == 0

    # #5. Define piecewise Green's function:
    G = sympy.Piecewise((Gm, Le(z,ξ)), (Gp, Ge(z,ξ)))
    
    # Check boundary conditions:
    @assert sympy.simplify(diff(G,z).subs(z,0).subs(ξ,-H//2)) == 0
    @assert sympy.simplify(G.subs(z,-H(x,y)).subs(ξ,-H//2)) == 0

    # Final sympy.simplify (to cancel constants). Avoid sympy.simplify in general because it's not always reproducible.
    G = sympy.simplify(G)
    return G
end ;

### Compute the velocity G's function:

In [6]:
Guv_sym = compute_Guv(uv_params,geometry_params) 
Guv = Guv_sym.subs(f,ϕ^2 * ϵ^2 * ν₀)

# Used for stress-driven part of the velocity:
Guv0 = Guv.subs(ξ,0)
Latex_Guv0 = (im * sympy.sqrt(im) * exp(-root_iϕ * z)/(ν₀ * ϵ^2 * ϕ * (1 + exp(2 * root_iϕ * H(x,y))))) *
(exp(2 * root_iϕ * (z + H(x,y))) - 1)
resid = Guv0 - Latex_Guv0
@assert resid == 0
println("0.1 LaTeX Guv0 expression matches.")

# Used for surface-pressure-driven part of the velocity:
tmp = sympy.integrate(sympy.expand(Guv), (ξ, -H(x,y), 0)).args[1].args[1]
max_obj = sympy.Max(z, -H(x, y))
Guv_int_wrt_ξ = tmp.subs(max_obj, z)
Latex_Guv_int_wrt_ξ = (im / (ν₀ * ϵ^2 * ϕ^2 * (1 + exp(2 * root_iϕ * H(x,y)))) ) *
(1 + exp(2 * root_iϕ * H(x,y)) - exp(root_iϕ * (z + H(x,y))) - exp(root_iϕ * (H(x,y) - z)))
resid = Guv_int_wrt_ξ - Latex_Guv_int_wrt_ξ
@assert resid == 0
println("0.2 LaTeX Guv integral wrt ξ expression matches.")

# Used for baroclinic-pressure-driven part of the velocity:
tmp = sympy.integrate(sympy.expand(Guv * exp(ψ * ξ)),(ξ,-H(x,y),0))
Guv_exp_psi_xi_int_wrt_ξ = tmp.subs(max_obj, z)
Latex_Guv_exp_psi_xi_int_wrt_ξ = (1/(ν₀ * ϵ^2 * (ψ^2 - im * ϕ^2))) * (
    ( im * sympy.sqrt(im) * ψ * exp(      ψ * H(x,y)) * (exp(2 * root_iϕ * (z + H(x,y))) - 1) -
                            ϕ * exp(root_iϕ * H(x,y)) * (exp(2 * root_iϕ *  z          ) + 1)
    ) * exp(-root_iϕ * z - ψ * H(x,y))
    /(ϕ * (1 + exp(2 * root_iϕ * H(x,y))))
    + exp(ψ * z)
)
resid = sympy.expand(Guv_exp_psi_xi_int_wrt_ξ - Latex_Guv_exp_psi_xi_int_wrt_ξ)
@assert resid == 0
println("0.3 LaTeX Guv * exp(ψ * z) integral matches.")

0.1 LaTeX Guv0 expression matches.
0.2 LaTeX Guv integral wrt ξ expression matches.
0.3 LaTeX Guv * exp(ψ * z) integral matches.


#### Function to solve the buoyancy equation using a Green's function:

In [7]:
function compute_Gb(b_params, geometry_params)
    # Setup symbols and parameters:
    κ, ϵ, γ = b_params
    H, z, ξ = geometry_params
    b       = SymFunction("b")
    A       = symbols("A",     real=true)                      # Unknown coefficient in the Green's function solution
    
    #0. Define the ODE for b(z):
    ode = Eq( - γ * b(z)     + ϵ^2 * diff(diff(κ *  b(z),z),z), 0)

    # 1. Solve for $G_- (z)$ on $-H \le z \le ξ$:
    bm = dsolve(ode, b(z), ics = Dict(diff(b(z),z).subs(z,-H)=>0)).rhs
    @assert sympy.simplify(ode.lhs.subs(b(z),bm)) == 0                  # Check solution
    @assert sympy.simplify(diff(bm,z).subs(z,-H) - 0) == 0              # Check Neumann BC at bottom
    bm_const = filter(x -> startswith(string(x), "C"), bm.free_symbols)
    bm = bm.subs(first(bm_const), A)                              # Replace constant with A so it doesn't conflict later

    # 2. Solve for $G_+(z)$ on  $ξ \le z \le 0$:
    bp = dsolve(ode, b(z), ics = Dict(b(0)=>0) ).rhs
    @assert sympy.simplify(ode.lhs.subs(b(z),bp)) == 0                  # Check solution
    @assert sympy.simplify(bp.subs(z,0)) == 0                           # Check Dirichlet BC at top

    # 3. Compute Wronskian $W(z)$:
    W = sympy.simplify(bm * diff(bp, z) - bp * diff(bm, z))

    # 4. Compute Green's function $G(z; ξ)$:
    Gm = sympy.simplify(bm * bp.subs(z,ξ) / (ϵ^2 * κ.subs(z,ξ) * W.subs(z,ξ)))
    Gp = sympy.simplify(bm.subs(z,ξ) * bp / (ϵ^2 * κ.subs(z,ξ) * W.subs(z,ξ)))

    # Check continuity and jump condition:
    @assert Gm.subs(z,ξ) - Gp.subs(z,ξ) == 0
    @assert sympy.simplify(diff(Gm, z).subs(z,ξ) - diff(Gp, z).subs(z,ξ)) + 1/(ϵ^2 * κ.subs(z,ξ)) == 0

    # Define piecewise Green's function:
    G = sympy.simplify(sympy.Piecewise((Gm, Le(z,ξ)), (Gp, Ge(z,ξ))))

    # Check boundary conditions are satisfied
    @assert sympy.simplify(diff(G,z).subs(z,-H).subs(ξ,-H//3).subs(H,1//2)) == 0
    @assert sympy.simplify(G.subs(z,0).subs(ξ,-H//3)) == 0

    return G
end

compute_Gb (generic function with 1 method)

### Compute the buoyancy Green's function:

In [8]:
Gb_sym  = compute_Gb(b_params,geometry_params)
Gb = Gb_sym.subs(γ,ψ^2 * ϵ^2 * κ₀)

⎧⎛     -2⋅ξ⋅ψ⎞ ⎛ 2⋅ψ⋅H(x, y)    -2⋅z⋅ψ⎞  ψ⋅(z + ξ)           
⎪⎝1 - ℯ      ⎠⋅⎝ℯ            + ℯ      ⎠⋅ℯ                    
⎪─────────────────────────────────────────────────  for z ≤ ξ
⎪                  2 ⎛ 2⋅ψ⋅H(x, y)    ⎞                      
⎪          2⋅κ₀⋅ψ⋅ϵ ⋅⎝ℯ            + 1⎠                      
⎨                                                            
⎪⎛     -2⋅z⋅ψ⎞ ⎛ 2⋅ψ⋅H(x, y)    -2⋅ξ⋅ψ⎞  ψ⋅(z + ξ)           
⎪⎝1 - ℯ      ⎠⋅⎝ℯ            + ℯ      ⎠⋅ℯ                    
⎪─────────────────────────────────────────────────  for z ≥ ξ
⎪                  2 ⎛ 2⋅ψ⋅H(x, y)    ⎞                      
⎩          2⋅κ₀⋅ψ⋅ϵ ⋅⎝ℯ            + 1⎠                      

#### Compute the buoyancy term explicitly for a linear relaxation buoyancy profile. This is to check and support the Latex derivation in `PlanetaryGeostrophySolutions_V0.5.tex`. The symbolic expressions themselves are used in `ExampleSolution_v0.6.ipynb`, which avoid SymPy by using `Symbolics.jl` instead. `Symbolics.jl` can't manage the integrations done by `Sympy` in this notebook.

In [9]:
# Compute $z$ integral of Gb for Latex notes:
Gb_int = SymPy.integrate(Gb,(z,z,0))
Gb_int = Gb_int.subs(max_obj, z)
Gb_int = sympy.simplify(Gb_int)
Latex_Gb_int_den = 2 * κ₀ * ψ^2 * ϵ^2 * (exp(2 * ψ * H(x,y)) + 1)
Latex_Gb_int_num = 
    exp(   ψ * ( z - ξ + 2 * H(x,y))) -
    exp(   ψ * ( z + ξ + 2 * H(x,y))) +
2 * exp(   ψ * ( ξ + 2 * H(x,y) ) ) + 
2 * exp( - ψ * ξ) +  
    exp(   ψ * ( ξ - z )) - 
    exp( - ψ * ( z + ξ )) -
    exp(   ψ * ( ξ + 2 * H(x,y) -  sympy.Max( z, ξ ))) - 
    exp( - ψ * ( ξ - 2 * H(x,y) -  sympy.Max( z, ξ ) )) -
    exp(   ψ * ( ξ - sympy.Max( z, ξ ) )) - 
    exp( - ψ * ( ξ - sympy.Max( z, ξ ) )) 

@assert sympy.simplify(Latex_Gb_int_num / Latex_Gb_int_den - Gb_int) == 0
println("1.1 LaTeX Gb integral matches.")

# Compute integrand for linear relaxation buoyancy profile explicitly:
tmp_integrand = sympy.expand(- Gb_int * ξ * α * γ)
tmp_integral  = sympy.integrate(tmp_integrand, (ξ, -H(x,y), 0))
tmp_integral  = sympy.simplify(tmp_integral.subs(max_obj, z))
Latex_Gb_int_int = z^2 +
    2*( 2 * exp(ψ * H(x,y)) - exp(ψ * (z + H(x,y))) - exp(ψ * (H(x,y) - z)) ) / (ψ^2 * (exp(2 * ψ * H(x,y)) + 1) )
Latex_Gb_int_int = (- α * γ) * Latex_Gb_int_int / (2 * κ₀ * ϵ^2 * ψ^2)
@assert sympy.simplify(Latex_Gb_int_int  - tmp_integral) == 0
println("1.2 LaTeX Gb double integral with linear buoyancy relaxation profile matches.")

# Differentiate Latex_Gb_int_int to get the baroclinic pressure gradient:
dGb_int_int_∂z = sympy.simplify(diff(Latex_Gb_int_int,x) + im*diff(Latex_Gb_int_int,y))
Latex_dGb_int_int_∂z = 
( ( (exp(ψ * H(x,y)) - exp(3 * ψ * H(x,y))) * (exp(ψ * z) - 2 + exp(- ψ * z)) ) / ( exp(2 * ψ * H(x,y)) + 1 )^2 ) * (diff(H(x,y),x) + im * diff(H(x,y),y))
Latex_dGb_int_int_∂z = (α * γ) * Latex_dGb_int_int_∂z / (κ₀ * ϵ^2 * ψ^3)
@assert sympy.simplify(dGb_int_int_∂z - Latex_dGb_int_int_∂z) == 0
println("1.3 LaTeX vertical integral of horizontal buoyancy derivative with linear buoyancy relaxation profile matches.")

# Construct the buoyancy related T(x) term (excludes the stress-driven part):
Txb_term_integrand = exp(-root_iϕ * (ξ - H(x,y))) * (1 + exp(2*root_iϕ * ξ)) / (1 + exp(2*root_iϕ * H(x,y)))
Txb_term_integrand = Txb_term_integrand * dGb_int_int_∂z.subs(z,ξ)
Txb_term = sympy.integrate(sympy.expand(Txb_term_integrand), (ξ, -H(x,y), 0))
Txb_term = sum(Txb_term.args[i] for i in 1:length(Txb_term.args)-1) + Txb_term.args[length(Txb_term.args)].args[1].args[1]    # Manually expand the integral result

Latex_Txb_term = α * γ * (exp((ψ + root_iϕ) * H(x,y)) - exp((3*ψ + root_iϕ) * H(x,y)))/
(κ₀ * ϵ^2 * ψ^3 * (1 + exp(2 * ψ * H(x,y)))^2 * (1 + exp(2 * root_iϕ * H(x,y)) ))
Latex_Txb_term = Latex_Txb_term * (sympy.diff(H(x,y),x) + im*sympy.diff(H(x,y),y))
Latex_int_result = (
((1 - exp(- (ψ - root_iϕ)*H(x,y)))/( ψ - root_iϕ)) +
((1 - exp(- (ψ + root_iϕ)*H(x,y)))/( ψ + root_iϕ)) +
(2/root_iϕ) * (exp(-root_iϕ * H(x,y)) - exp(root_iϕ * H(x,y))) +
((1 - exp(  (ψ + root_iϕ)*H(x,y)))/(-ψ - root_iϕ)) + 
((1 - exp(  (ψ - root_iϕ)*H(x,y)))/(-ψ + root_iϕ)) 
)
# Check intermediate analytic Latex integral:
inter1 = exp(- (ψ - root_iϕ)*ξ) + exp(- (ψ + root_iϕ)*ξ) - 2*(exp(-root_iϕ*ξ) + exp(+root_iϕ*ξ)) + exp((ψ + root_iϕ)*ξ) + exp((ψ - root_iϕ)*ξ)
inter2 = sympy.integrate(inter1, (ξ,-H(x,y),0))
@assert inter2 - Latex_int_result == 0
Latex_Txb_term = Latex_Txb_term * Latex_int_result
# Substitute specific parameter values to check equality, otherwise SymPy takes a very long time...
tmp = (sympy.expand(Latex_Txb_term) - sympy.expand(Txb_term)).subs(sample_params).doit()  # For easier comparison
tmp = sympy.simplify(tmp)
@assert tmp == 0
println("1.4 LaTeX T(x) buoyancy term with linear buoyancy relaxation profile matches.")

# Construct the buoyancy related B(x) term (excludes the stress-driven part):
Bxb_term_integrand = exp(-root_iϕ * ξ) * (exp(root_iϕ * ξ) - exp(root_iϕ * H(x,y))) * (exp(root_iϕ * (ξ + H(x,y))) - 1)
Bxb_term_integrand = Bxb_term_integrand * dGb_int_int_∂z.subs(z,ξ)
Bxb_term = sympy.integrate(sympy.expand(Bxb_term_integrand), (ξ, -H(x,y), 0))

Latex_Bxb_term = α * γ * (exp(ψ * H(x,y)) - exp(3 * ψ * H(x,y)))/(κ₀ * ϵ^2 * ψ^3 * (1 + exp(2 * ψ * H(x,y)))^2)
Latex_Bxb_term = Latex_Bxb_term * (sympy.diff(H(x,y),x) + im*sympy.diff(H(x,y),y))
Latex_int_result = exp(root_iϕ * H(x,y)) * (
((1 - exp(- (ψ - root_iϕ)*H(x,y)))/( ψ - root_iϕ)) +
((1 - exp(- (ψ + root_iϕ)*H(x,y)))/( ψ + root_iϕ)) +
(2/root_iϕ) * (exp(-root_iϕ * H(x,y)) - exp(root_iϕ * H(x,y))) +
((1 - exp(  (ψ + root_iϕ)*H(x,y)))/(-ψ - root_iϕ)) + 
((1 - exp(  (ψ - root_iϕ)*H(x,y)))/(-ψ + root_iϕ)) +
((exp((-ψ + root_iϕ) * H(x,y)) - exp((ψ + root_iϕ) * H(x,y)))/ψ)
)
Latex_int_result = Latex_int_result + (1/ψ)*(exp(-ψ * H(x,y)) - exp(ψ * H(x,y))) + 2*H(x,y)*(1 + exp(2*root_iϕ * H(x,y)))
# Check intermediate analytic Latex integral:
inter1 = (1 - exp(root_iϕ * (H(x,y) - ξ))) * (exp(root_iϕ * (H(x,y) + ξ)) - 1) * (exp(ψ * ξ) - 2 + exp(- ψ * ξ))
inter2 = sympy.integrate(sympy.expand(inter1), (ξ,-H(x,y),0))
@assert inter2 - Latex_int_result == 0
Latex_Bxb_term = Latex_Bxb_term * Latex_int_result
# Substitute specific parameter values to check equality, otherwise SymPy takes a very long time...
tmp = (sympy.expand(Latex_Bxb_term) - Bxb_term).subs(sample_params).doit()  # For easier comparison
tmp = sympy.simplify(tmp)
@assert tmp == 0
println("1.5 LaTeX B(x) buoyancy term with linear buoyancy relaxation profile matches.")

1.1 LaTeX Gb integral matches.
1.2 LaTeX Gb double integral with linear buoyancy relaxation profile matches.
1.3 LaTeX vertical integral of horizontal buoyancy derivative with linear buoyancy relaxation profile matches.
1.4 LaTeX T(x) buoyancy term with linear buoyancy relaxation profile matches.
1.5 LaTeX B(x) buoyancy term with linear buoyancy relaxation profile matches.


### Symbolic computation of buoyancy $b$

In [ ]:
# Compute buoyancy field b(z):
b_sym = SymPy.integrate(sympy.expand(-Gb * γ * B(x,y,ξ)), (ξ,-H(x,y),0) )
b = sympy.simplify(b_sym.subs(max_obj,z))

# Check that the final expression for b satisfies the original differential equation:
tmp1 = ϵ^2 * diff(diff(κ * b,z),z) - γ * b
tmp1 = sympy.simplify(tmp1.subs(ψ,sqrt(γ/κ₀)/ϵ))
@assert tmp1 + γ * B(x,y,z) == 0
println("1.6 Final expression for b(z) satisfies the original differential equation.")

### Define parameter values:

In [ ]:
# Define domain depth:
H_val(x,y)    = 1.0 - x^2 - y^2

# Define relaxation buoyancy field:
# B_val(x,y,ξ)  = α * ξ
# B_val(x,y,ξ)  = α * ξ^2
n   = symbols("n",   integer=true)
B_val(x,y,ξ)  = α * ξ^n

α_val = 0.3
B_valH(x, y)  = B_val(x,y,-H_val(x,y) )

# Define dictionaries for substitutions:
B_values     = Dict(
    B(x, y, ξ)=>B_val(x, y, ξ), 
    B(x, y, z)=>B_val(x, y, z), 
    B(x, y, -H(x,y))=>B_valH(x,y)
    )

Dict{Sym{PyCall.PyObject}, Sym{PyCall.PyObject}} with 3 entries:
  B(x, y, ξ)        => α*ξ
  B(x, y, z)        => z*α
  B(x, y, -H(x, y)) => α*(x^2 + y^2 - 1.0)

#### Compute buoyancy expression for Latex notes and check it:

In [16]:
Latex_b = ((- α * γ)) / (κ₀ * ψ^2 * ϵ^2) * ( 
(exp(ψ * H(x,y)) * (exp(ψ * z) - exp(-ψ * z))) /(ψ * (1 + exp(2 * ψ * H(x,y))))
- z
)
@assert b.subs(B_values) - Latex_b == 0
println("2.7 LaTeX b(x) buoyancy expression with linear buoyancy relaxation profile matches.")

2.7 LaTeX b(x) buoyancy expression with linear buoyancy relaxation profile matches.
